# PDF to Markdown with Qwen3-VL (via OpenRouter)

Spike notebook to test converting PDF pages to formatted markdown using Qwen3-VL vision model.

**Workflow:**
1. Convert all PDF pages to PNG images (saved to disk)
2. Process pages in batches via Qwen3-VL
3. Save each batch's markdown to separate files

In [11]:
import base64
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path

import pymupdf  # type: ignore[import-untyped]
import requests
from dotenv import find_dotenv, load_dotenv

sys.path.append("..")
load_dotenv(find_dotenv(".env"), override=True)

True

## Configuration

In [12]:
# Paths
PDF_PATH = Path("../../test_data/qi-ti-yuan-liu-a.pdf")
OUTPUT_DIR = Path("../../test_data/output")
IMAGES_DIR = OUTPUT_DIR / "images"
MARKDOWN_DIR = OUTPUT_DIR / "markdown"

# Create output directories
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

# API config
_api_key = os.getenv("OPENROUTER_API_KEY")
if _api_key is None:
    raise ValueError("OPENROUTER_API_KEY not found in environment")
API_KEY: str = _api_key

MODEL = "qwen/qwen3-vl-235b-a22b-instruct"
BATCH_SIZE = 10  # pages per API call

print(f"PDF: {PDF_PATH}")
print(f"Output images: {IMAGES_DIR}")
print(f"Output markdown: {MARKDOWN_DIR}")
print(f"Model: {MODEL}")
print(f"Batch size: {BATCH_SIZE} pages")

PDF: ../../test_data/qi-ti-yuan-liu-a.pdf
Output images: ../../test_data/output/images
Output markdown: ../../test_data/output/markdown
Model: qwen/qwen3-vl-235b-a22b-instruct
Batch size: 10 pages


## Function Definitions

In [13]:
def pdf_to_images_on_disk(pdf_path: Path, output_dir: Path, dpi: int = 150) -> list[Path]:
    """
    Convert all PDF pages to PNG images and save to disk.
    
    Args:
        pdf_path: Path to the PDF file
        output_dir: Directory to save images
        dpi: Resolution for rendering (150 = 2x scale)
    
    Returns:
        List of paths to saved image files
    """
    doc = pymupdf.open(pdf_path)
    image_paths: list[Path] = []
    
    # Calculate zoom factor for desired DPI (72 is PDF base DPI)
    zoom = dpi / 72
    mat = pymupdf.Matrix(zoom, zoom)
    
    print(f"Converting {len(doc)} pages to images at {dpi} DPI...")
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        pix = page.get_pixmap(matrix=mat)
        
        # Save as PNG
        image_path = output_dir / f"page_{page_num + 1:04d}.png"
        pix.save(str(image_path))
        image_paths.append(image_path)
        
        if (page_num + 1) % 10 == 0 or page_num == len(doc) - 1:
            print(f"  Saved page {page_num + 1}/{len(doc)}")
    
    doc.close()
    print(f"Done! {len(image_paths)} images saved to {output_dir}")
    return image_paths

In [14]:
@dataclass
class ConversionResult:
    """Result of PDF to markdown conversion with cost tracking."""
    markdown: str
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int
    cost_usd: float
    raw_response: dict = field(default_factory=dict, repr=False)


def images_to_markdown(
    image_paths: list[Path],
    api_key: str,
    model: str = "qwen/qwen3-vl-235b-a22b-instruct",
) -> ConversionResult:
    """
    Send images to Qwen3-VL via OpenRouter and get markdown output.
    
    Args:
        image_paths: List of paths to PNG images
        api_key: OpenRouter API key
        model: Model identifier
    
    Returns:
        ConversionResult with markdown and usage stats
    """
    # Build content array with all images
    content = []
    for img_path in image_paths:
        with open(img_path, "rb") as f:
            img_b64 = base64.b64encode(f.read()).decode("utf-8")
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{img_b64}"
            }
        })
    
    # Add the instruction text
    content.append({
        "type": "text",
        "text": """Convert these PDF pages to well-formatted markdown. 
Preserve the document structure including:
- Headings and subheadings
- Paragraphs
- Bullet points and numbered lists
- Tables (use markdown table format)
- Any emphasized or bold text

Output only the markdown content, no explanations."""
    })
    
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "messages": [
                {
                    "role": "user",
                    "content": content
                }
            ],
            "max_tokens": 16000,
        },
        timeout=300,
    )
    
    response.raise_for_status()
    result = response.json()
    
    # Extract usage stats
    usage = result.get("usage", {})
    prompt_tokens = usage.get("prompt_tokens", 0)
    completion_tokens = usage.get("completion_tokens", 0)
    total_tokens = usage.get("total_tokens", prompt_tokens + completion_tokens)
    
    # OpenRouter cost - check multiple possible locations
    cost_usd = (
        usage.get("total_cost") or 
        usage.get("cost") or 
        result.get("total_cost") or
        0.0
    )
    
    return ConversionResult(
        markdown=result["choices"][0]["message"]["content"],
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        total_tokens=total_tokens,
        cost_usd=float(cost_usd),
        raw_response=result,
    )

In [15]:
@dataclass 
class BatchResult:
    """Result for a batch of pages."""
    batch_num: int
    start_page: int
    end_page: int
    markdown_path: Path
    result: ConversionResult


def process_pdf_in_batches(
    image_paths: list[Path],
    output_dir: Path,
    api_key: str,
    model: str,
    batch_size: int = 10,
) -> list[BatchResult]:
    """
    Process images in batches and save markdown files.
    
    Args:
        image_paths: List of image paths (in order)
        output_dir: Directory to save markdown files
        api_key: OpenRouter API key
        model: Model identifier
        batch_size: Number of pages per batch
    
    Returns:
        List of BatchResult objects
    """
    results: list[BatchResult] = []
    total_pages = len(image_paths)
    num_batches = (total_pages + batch_size - 1) // batch_size
    
    total_cost = 0.0
    total_prompt_tokens = 0
    total_completion_tokens = 0
    
    print(f"Processing {total_pages} pages in {num_batches} batch(es)...")
    print("=" * 60)
    
    for batch_num in range(num_batches):
        start_idx = batch_num * batch_size
        end_idx = min(start_idx + batch_size, total_pages)
        batch_images = image_paths[start_idx:end_idx]
        
        print(f"\nBatch {batch_num + 1}/{num_batches}: pages {start_idx + 1}-{end_idx}")
        print("-" * 40)
        
        # Convert batch to markdown
        result = images_to_markdown(batch_images, api_key, model)
        
        # Save markdown file
        md_filename = f"pages_{start_idx + 1:04d}-{end_idx:04d}.md"
        md_path = output_dir / md_filename
        md_path.write_text(result.markdown, encoding="utf-8")
        
        # Track totals
        total_cost += result.cost_usd
        total_prompt_tokens += result.prompt_tokens
        total_completion_tokens += result.completion_tokens
        
        # Print batch stats
        print(f"  Prompt tokens:     {result.prompt_tokens:,}")
        print(f"  Completion tokens: {result.completion_tokens:,}")
        print(f"  Cost:              ${result.cost_usd:.6f} USD")
        print(f"  Saved to:          {md_path}")
        
        results.append(BatchResult(
            batch_num=batch_num + 1,
            start_page=start_idx + 1,
            end_page=end_idx,
            markdown_path=md_path,
            result=result,
        ))
    
    # Print summary
    print("\n" + "=" * 60)
    print("TOTAL SUMMARY")
    print("=" * 60)
    print(f"Pages processed:     {total_pages}")
    print(f"Batches:             {num_batches}")
    print(f"Total prompt tokens: {total_prompt_tokens:,}")
    print(f"Total completion:    {total_completion_tokens:,}")
    print(f"Total cost:          ${total_cost:.6f} USD")
    print(f"Cost per page:       ${total_cost / total_pages:.6f} USD")
    
    return results

## Step 1: Convert PDF to Images

In [16]:
# Convert all PDF pages to images
image_paths = pdf_to_images_on_disk(PDF_PATH, IMAGES_DIR, dpi=150)

Converting 234 pages to images at 150 DPI...
  Saved page 10/234
  Saved page 20/234
  Saved page 30/234
  Saved page 40/234
  Saved page 50/234
  Saved page 60/234
  Saved page 70/234
  Saved page 80/234
  Saved page 90/234
  Saved page 100/234
  Saved page 110/234
  Saved page 120/234
  Saved page 130/234
  Saved page 140/234
  Saved page 150/234
  Saved page 160/234
  Saved page 170/234
  Saved page 180/234
  Saved page 190/234
  Saved page 200/234
  Saved page 210/234
  Saved page 220/234
  Saved page 230/234
  Saved page 234/234
Done! 234 images saved to ../../test_data/output/images


In [17]:
# Show first few images
print(f"Total images: {len(image_paths)}")
for p in image_paths[:5]:
    print(f"  {p.name}")
if len(image_paths) > 5:
    print(f"  ... and {len(image_paths) - 5} more")

Total images: 234
  page_0001.png
  page_0002.png
  page_0003.png
  page_0004.png
  page_0005.png
  ... and 229 more


## Step 2: Convert Images to Markdown (First Batch Only)

In [18]:
# Process first batch only (for testing)
first_batch = image_paths[:BATCH_SIZE]
print(f"Processing first {len(first_batch)} pages...")

result = images_to_markdown(first_batch, API_KEY, MODEL)

# Save markdown
md_path = MARKDOWN_DIR / f"pages_0001-{len(first_batch):04d}.md"
md_path.write_text(result.markdown, encoding="utf-8")

print(f"\nResult:")
print(f"  Prompt tokens:     {result.prompt_tokens:,}")
print(f"  Completion tokens: {result.completion_tokens:,}")
print(f"  Total tokens:      {result.total_tokens:,}")
print(f"  Cost:              ${result.cost_usd:.6f} USD")
print(f"  Saved to:          {md_path}")

Processing first 10 pages...

Result:
  Prompt tokens:     9,617
  Completion tokens: 423
  Total tokens:      10,040
  Cost:              $0.002488 USD
  Saved to:          ../../test_data/output/markdown/pages_0001-0010.md


In [19]:
# Debug: print raw response to find cost field
print("Raw usage from API response:")
print(result.raw_response.get("usage", {}))

Raw usage from API response:
{'prompt_tokens': 9617, 'completion_tokens': 423, 'total_tokens': 10040, 'cost': 0.00248798, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 1, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': None, 'upstream_inference_prompt_cost': 0.00211574, 'upstream_inference_completions_cost': 0.00037224}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0}}


In [20]:
# Display rendered markdown
from IPython.display import Markdown, display

display(Markdown(result.markdown))

```markdown
无体源流

米晶子 编著

深圳报业集团出版社

---

“炁”化三清，“體”能載道，“源”乃先天道统，“流”為老君法脉，故曰《炁體源流》。

—— 全真龍門派第二十一代 张至顺 颁米晶子

---

无体源流

米晶子 编著

深圳报业集团出版社出版发行

(518009 深圳市深南大道 6008 号)

三河市华晨印务有限公司印制 新华书店经销

2012 年 12 月第 1 版 2012 年 12 月第 1 次印刷

开本: 787mm × 1092mm 1/16

印张: 32.25 字数: 140 千字

ISBN 978-7-80709-475-3 定价: 118.00 元

深报版图书版权所有，侵权必究。

深报版图书凡是有印装质量问题，请随时向承印厂调换。

---

改变，从心开始

立品图书·自觉·觉他

www.tobebooks.net

出品

---

遍地金莲一起开就在目前

---

师姓李名耳字聃号老子

---

师姓吕名岩字洞宾号纯阳子

---

太乙雷声应化天尊王善王灵官

---

```

Note: The image on page 9 is a grainy, low-resolution photograph of an elderly man with a long white beard, which cannot be accurately described or transcribed into markdown text. It is omitted from the markdown output as per the instruction to preserve document structure and content, and since no textual information is present in the image itself.

## Step 3: Process All Pages (Optional)

Uncomment and run to process all pages in batches.

In [ ]:
# Process ALL pages in batches
all_results = process_pdf_in_batches(
    image_paths=image_paths,
    output_dir=MARKDOWN_DIR,
    api_key=API_KEY,
    model=MODEL,
    batch_size=BATCH_SIZE,
)